# **Ensemble Prediction and Evaluation Across Targets**

This notebook constructs ensemble models by combining predictions from the Chemprop deep learning model with the two best-performing traditional machine learning (TradML) models for each biological target. The candidate TradML models were previously evaluated, and the top-performing models were manually selected based on their predictive performance.


For each target, the notebook loads the test prediction files containing outputs from individual models (e.g., Random Forest, SVR, XGBoost, or MLP). Ensemble predictions are then generated using a simple equal-weight averaging strategy across the selected models.

The ensemble predictions are evaluated against the true pIC50 values using R² and RMSE metrics. The resulting ensemble predictions are appended to the original prediction files, and a summary table of ensemble performance across all targets is saved as ensemble_summary_results.csv.

This step enables comparison between individual models and ensemble models, supporting the identification of improved predictive performance through model combination.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Define ensemble configuration

In [ ]:
ensemble_config = {
    "5-HT6": {
        "ensemble_name": "RF+SVR+Chemprop",
        "pred_cols": ["pred_RandomForest", "pred_SVR", "pred_Chemprop"]
    },
    "ache": {
        "ensemble_name": "XGBoost+SVR+Chemprop",
        "pred_cols": ["pred_XGBoost", "pred_SVR", "pred_Chemprop"]
    },
    "bace1": {
        "ensemble_name": "XGBoost+SVR+Chemprop",
        "pred_cols": ["pred_XGBoost", "pred_SVR", "pred_Chemprop"]
    },
    "buche": {
        "ensemble_name": "XGBoost+SVR+Chemprop",
        "pred_cols": ["pred_XGBoost", "pred_SVR", "pred_Chemprop"]
    },
    "esr1": {
        "ensemble_name": "SVR+MLP+Chemprop",
        "pred_cols": ["pred_SVR","pred_MLPRegressor",  "pred_Chemprop"]
    },
    "gsk-3beta": {
        "ensemble_name": "XGBoost+SVR+Chemprop",
        "pred_cols": ["pred_XGBoost", "pred_SVR", "pred_Chemprop"]
    },
    "mao-b": {
        "ensemble_name": "XGBoost+SVR+Chemprop",
        "pred_cols": ["pred_XGBoost", "pred_SVR", "pred_Chemprop"]
    }
}


## Metric helpers

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


## Ensemble + evaluation loop (per target)

In [ ]:
import pandas as pd
import os

results = []

pred_dir = "/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/trad-ml-predictions"

for target, cfg in ensemble_config.items():

    test_fp = f"{pred_dir}/{target}_test_preds_tradML.csv"
    df = pd.read_csv(test_fp)

    y_true = df["pIC50"].values
    pred_cols = cfg["pred_cols"]

    # Safety check
    missing = [c for c in pred_cols if c not in df.columns]
    if missing:
        print(f"[SKIP] {target} missing columns: {missing}")
        continue

    # Ensemble prediction (equal weight)
    ens_pred = df[pred_cols].mean(axis=1)

    # Metrics
    r2 = r2_score(y_true, ens_pred)
    err = rmse(y_true, ens_pred)

    # Store results
    results.append({
        "Target": target,
        "Ensemble": cfg["ensemble_name"],
        "Models_Combined": ", ".join(pred_cols),
        "Test_R2": round(r2, 4),
        "Test_RMSE": round(err, 4)
    })

    #  save ensemble predictions back to file
    df[f"pred_{cfg['ensemble_name']}"] = ens_pred
    df.to_csv(test_fp, index=False)

    print(f"[OK] {target} | R2={r2:.3f} | RMSE={err:.3f}")


[OK] 5-HT6 | R2=0.626 | RMSE=0.607
[OK] ache | R2=0.704 | RMSE=0.713
[OK] bace1 | R2=0.705 | RMSE=0.649
[OK] buche | R2=0.769 | RMSE=0.609
[OK] esr1 | R2=0.724 | RMSE=0.756
[OK] gsk-3beta | R2=0.682 | RMSE=0.713
[OK] mao-b | R2=0.673 | RMSE=0.706


In [ ]:
import os
import pandas as pd

# 1) Confirm folder
print("pred_dir =", pred_dir)
print("Exists?", os.path.exists(pred_dir))

# 2) Create summary df from 'results'
ensemble_results_df = pd.DataFrame(results)

# 3) Save
out_fp = os.path.join(pred_dir, "ensemble_summary_results.csv")
ensemble_results_df.to_csv(out_fp, index=False)

print("Saved:", out_fp)
print("\nPreview:")
display(ensemble_results_df)


pred_dir = /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/trad-ml-predictions
Exists? True
Saved: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/trad-ml-predictions/ensemble_summary_results.csv

Preview:


,Target,Ensemble,Models_Combined,Test_R2,Test_RMSE
0,5-HT6,RF+SVR+Chemprop,"pred_RandomForest, pred_SVR, pred_Chemprop",0.6259,0.6070
1,ache,XGBoost+SVR+Chemprop,"pred_XGBoost, pred_SVR, pred_Chemprop",0.7038,0.7129
2,bace1,XGBoost+SVR+Chemprop,"pred_XGBoost, pred_SVR, pred_Chemprop",0.7053,0.6490
3,buche,XGBoost+SVR+Chemprop,"pred_XGBoost, pred_SVR, pred_Chemprop",0.7693,0.6091
4,esr1,SVR+MLP+Chemprop,"pred_SVR, pred_MLPRegressor, pred_Chemprop",0.7240,0.7563
5,gsk-3beta,XGBoost+SVR+Chemprop,"pred_XGBoost, pred_SVR, pred_Chemprop",0.6823,0.7128
6,mao-b,XGBoost+SVR+Chemprop,"pred_XGBoost, pred_SVR, pred_Chemprop",0.6728,0.7058
